# Rooting by duplication-loss-coalescence reconciliation

`tree.mod.root_on_minimal_dlc()` roots a gene tree by testing every rootable edge on the unrooted gene-tree topology and choosing the edge with the lowest weighted duplication-loss-coalescence (DLC) reconciliation cost against a rooted species tree.

Use this method when you have a gene tree, a rooted species tree, and a mapping from gene tips to species tips.


In [ ]:
import toytree


## Main function

`tree.mod.root_on_minimal_dlc(species_tree, imap, return_stats=False, store_scores=False, weight_duplications=3.0, weight_losses=1.0, weight_coalescences=0.0)`

The species tree must already be rooted. In practice it is safest to pass `imap` explicitly, especially when the gene tree contains duplicated genes per species.


In [ ]:
species_tree = toytree.tree("(((A,B),C),D);")
gene_tree = toytree.tree("((((a1,a2),b1),c1),d1);")
imap = {"a1": "A", "a2": "A", "b1": "B", "c1": "C", "d1": "D"}


## A simple DLC-rooting example

The species tree and gene tree below use the same set of sampled lineages, but the gene tree contains a duplication in species `A`. The rooting algorithm evaluates every candidate root edge on the gene tree and selects the one with the lowest weighted reconciliation cost.


In [ ]:
c, a, m = toytree.mtree([species_tree, gene_tree.unroot()]).draw(
    layout="d",
    node_sizes=12,
    node_labels="idx",
    tip_labels=True,
    width=520,
    height=240,
)
a[0].label.text = "rooted species tree"
a[1].label.text = "unrooted gene tree"
c


In [ ]:
rooted_gene = gene_tree.mod.root_on_minimal_dlc(species_tree, imap)
rooted_gene.draw(layout="d", node_labels="idx", node_sizes=12);


## Inspect the selected root with `return_stats=True`

If you request stats, the function returns `(tree, stats)`. The stats dictionary includes the chosen edge, the weighted score, the event counts on that best solution, and a full score table for all candidate edges.


In [ ]:
rooted_gene, stats = gene_tree.mod.root_on_minimal_dlc(
    species_tree,
    imap,
    return_stats=True,
)

stats["best_edge_idx"], stats["best_score"], stats["best_counts"]


In [ ]:
stats["score_table"]


## Store per-edge scores with `store_scores=True`

When `store_scores=True`, the returned tree stores two edge features:

- `DLC`: the weighted reconciliation score for rooting on each edge.
- `DLC_root_prob`: a simple probability mass spread across the tied best edges.


In [ ]:
scored_gene = gene_tree.mod.root_on_minimal_dlc(
    species_tree,
    imap,
    store_scores=True,
)

scored_gene.get_node_data()[["name", "DLC", "DLC_root_prob"]]


In [ ]:
c, a, m = scored_gene.draw(layout="d", width=480, node_sizes=10, node_labels="idx")
scored_gene.annotate.add_edge_labels(a, "DLC_root_prob", mask=False, font_size=11)
c


## Change the objective with the weight parameters

The default objective emphasizes duplications more heavily than losses and ignores coalescences. You can change that balance by adjusting the three weight parameters. In this example the preferred root stays the same, but the edge scores and ranking criterion change.


In [ ]:
_, default_stats = gene_tree.mod.root_on_minimal_dlc(
    species_tree,
    imap,
    return_stats=True,
)
_, alt_stats = gene_tree.mod.root_on_minimal_dlc(
    species_tree,
    imap,
    return_stats=True,
    weight_duplications=0.5,
    weight_losses=3.0,
    weight_coalescences=0.0,
)

comparison = default_stats["score_table"][["edge_idx", "score"]].rename(
    columns={"score": "default_score"}
).merge(
    alt_stats["score_table"][["edge_idx", "score"]].rename(
        columns={"score": "loss_weighted_score"}
    ),
    on="edge_idx",
)
comparison


## Related APIs

- [`mod-rooting-outgroup.md`](mod-rooting-outgroup.md) for manual rooting.
- [`mod-rooting-branch-length.md`](mod-rooting-branch-length.md) for midpoint, balanced midpoint, and MAD rooting.
